In [ ]:
import os
import torch
import torchaudio
from torch.utils.data import Dataset , DataLoader
import json
from transformers import ASTFeatureExtractor ,ASTForAudioClassification

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
folders = ['Layal_vs_Fery' , 'Layal_vs_Hsu' , 'Layal_vs_Martin']
files_list = []

for folder in folders :
    paths = os.listdir(os.path.join('/content/drive/MyDrive' ,'dataset', folder , 'audios'))
    full_paths = [os.path.join('/content/drive/MyDrive' , 'dataset' , folder , 'audios' , path) for path in paths ]
    files_list.extend(full_paths)


In [ ]:
files_list

['/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/frame_259_no_event.wav',
 '/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/frame_444_no_event.wav',
 '/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/frame_185_no_event.wav',
 '/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/frame_481_no_event.wav',
 '/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/frame_370_no_event.wav',
 '/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/frame_74_no_event.wav',
 '/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/frame_296_no_event.wav',
 '/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/frame_148_no_event.wav',
 '/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/frame_37_no_event.wav',
 '/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/frame_111_no_event.wav',
 '/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/frame_407_no_event.wav',
 '/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/frame_333_no_event.wav',
 '/content/drive/MyDrive/dataset/Layal_vs_Fery/audios/

In [ ]:
id2label={
        0: "ball_hit",
        1: "ball_bounced",
        2: "no_event"
    },
label2id={
        "ball_hit": 0,
        "ball_bounced": 1,
        "no_event": 2
    }

In [ ]:
feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

In [ ]:
import torch
import torchaudio
from torch.utils.data import Dataset


class AudioDataset(Dataset):

    def __init__(self, file_paths):
        self.file_paths = file_paths

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):

        path = self.file_paths[idx]

        filename = path.split("/")[-1].split(".")[0]

        # frame_7718_ball_bounced -> ball_bounced
        label = "_".join(filename.split("_")[-2:])
        label = label2id[label]

        waveform, sr = torchaudio.load(path)

        # stereo -> mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0)

        else:
            waveform = waveform.squeeze(0)

        return {
            "waveform": waveform,
            "sampling_rate": sr,
            "labels": label
        }

In [ ]:
from sklearn.model_selection import train_test_split

train_paths, val_paths = train_test_split(
    files_list,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

In [ ]:
train_dataset = AudioDataset(train_paths)
val_dataset = AudioDataset(val_paths)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

import torchaudio
import torch


class ASTDataCollator:

    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def __call__(self, batch):

        waveforms = []
        labels = []

        for sample in batch:

            waveform = sample["waveform"]
            sr = sample["sampling_rate"]

            # AST expects 16 kHz
            if sr != 16000:

                waveform = torchaudio.functional.resample(
                    waveform,
                    sr,
                    16000
                )

            waveforms.append(waveform.numpy())
            labels.append(sample["labels"])

        features = self.feature_extractor(
            waveforms,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )

        features["labels"] = torch.tensor(labels)

        return features

In [ ]:
from transformers import ASTForAudioClassification

id2label = {
    0: "ball_hit",
    1: "ball_bounced",
    2: "no_event"
}

model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=3,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `527`.


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                         
------------------------+----------+-----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([3])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")


def compute_metrics(eval_pred):

    predictions = np.argmax(
        eval_pred.predictions,
        axis=1
    )

    acc = accuracy.compute(
        predictions=predictions,
        references=eval_pred.label_ids
    )

    f1_score = f1.compute(
        predictions=predictions,
        references=eval_pred.label_ids,
        average="macro"
    )

    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./ast-tennis",

    num_train_epochs=15,

    learning_rate=1e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=20,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    save_total_limit=2,

    report_to="none",

    remove_unused_columns=False # Added this line to prevent removal of 'waveform'
)

In [ ]:
from transformers import Trainer

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    data_collator=ASTDataCollator(feature_extractor),

    processing_class=feature_extractor,

    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.572816,0.810345,0.808352
2,0.851496,0.279999,0.913793,0.914263
3,0.304106,0.177855,0.931034,0.926191
4,0.094591,0.153896,0.931034,0.926226
5,0.094591,0.089938,0.948276,0.942614
6,0.025571,0.068268,0.982759,0.981495
7,0.006152,0.079274,0.965517,0.962274
8,0.001699,0.158564,0.931034,0.927391
9,0.001699,0.110346,0.982759,0.981495
10,0.000797,0.114227,0.982759,0.981495


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['audio_spectrogram_transformer.layers.0.attention.q_proj.weight', 'audio_spectrogram_transformer.layers.0.attention.q_proj.bias', 'audio_spectrogram_transformer.layers.0.attention.k_proj.weight', 'audio_spectrogram_transformer.layers.0.attention.k_proj.bias', 'audio_spectrogram_transformer.layers.0.attention.v_proj.weight', 'audio_spectrogram_transformer.layers.0.attention.v_proj.bias', 'audio_spectrogram_transformer.layers.0.attention.o_proj.weight', 'audio_spectrogram_transformer.layers.0.attention.o_proj.bias', 'audio_spectrogram_transformer.layers.0.layernorm_before.weight', 'audio_spectrogram_transformer.layers.0.layernorm_before.bias', 'audio_spectrogram_transformer.layers.0.layernorm_after.weight', 'audio_spectrogram_transformer.layers.0.layernorm_after.bias', 'audio_spectrogram_transformer.layers.0.mlp.fc1.weight', 'audio_spectrogram_transformer.layers.0.mlp.fc1.bias', 'audio_spectrogram_transformer.layers.

TrainOutput(global_step=225, training_loss=0.11437210517521534, metrics={'train_runtime': 670.3523, 'train_samples_per_second': 5.147, 'train_steps_per_second': 0.336, 'total_flos': 2.338528840777728e+17, 'train_loss': 0.11437210517521534, 'epoch': 15.0})

In [ ]:
pip install huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/ahmedmohamed55/ast-tennis/commit/77f53c8443162e0d882c5d410c155469df96f37e', commit_message='End of training', commit_description='', oid='77f53c8443162e0d882c5d410c155469df96f37e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ahmedmohamed55/ast-tennis', endpoint='https://huggingface.co', repo_type='model', repo_id='ahmedmohamed55/ast-tennis'), pr_revision=None, pr_num=None)